# Muon Ablation Demo

This notebook reproduces a minimal end-to-end workflow:

1. Clone the repository
2. Install dependencies with `uv`
3. Run **6 training setups** — 3 architectures × 2 optimizers (AdamW, Muon)
4. Review results and suggested follow-up experiments

> **Tip:** Demo overrides use small data and few epochs. For real runs, see [README.md](README.md).

In [ ]:
import os
import subprocess
from pathlib import Path

# --- configure clone target ---
REPO_URL = "https://github.com/MichaelNotDeveloper/muon_ablation.git"
BRANCH = "main"  # or your branch name
CLONE_DIR = Path.home() / "muon_ablation_demo"

# Set True if you already cloned / are developing inside the repo
SKIP_CLONE = False
USE_LOCAL_REPO = Path.cwd().name == "muon_ablation" and (Path.cwd() / "train.py").exists()

if USE_LOCAL_REPO and not SKIP_CLONE:
    CLONE_DIR = Path.cwd()
    print("Detected local repo; using current directory:", CLONE_DIR)
else:
    print(f"Target clone directory: {CLONE_DIR}")

In [ ]:
if USE_LOCAL_REPO and CLONE_DIR == Path.cwd():
    print("Skipping clone; already in repo.")
elif CLONE_DIR.exists():
    print(f"Directory already exists, pulling latest changes: {CLONE_DIR}")
    subprocess.run(["git", "-C", str(CLONE_DIR), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(CLONE_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(CLONE_DIR), "pull", "origin", BRANCH], check=True)
else:
    print(f"Cloning {REPO_URL} -> {CLONE_DIR}")
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, REPO_URL, str(CLONE_DIR)],
        check=True,
    )

os.chdir(CLONE_DIR)
print("Working directory:", Path.cwd())

In [ ]:
# Install dependencies (requires uv: pip install uv)
subprocess.run(["uv", "sync"], check=True)
print("Dependencies installed.")

## Demo training grid

| # | Config | Optimizer | Run name |
|---|--------|-----------|----------|
| 1 | `lstm` | AdamW | `lstm_adamw` |
| 2 | `lstm` | Muon | `lstm_muon` |
| 3 | `transformer` | AdamW | `transformer_adamw` |
| 4 | `transformer` | Muon | `transformer_muon` |
| 5 | `nanogpt` | AdamW | `nanogpt_adamw` |
| 6 | `nanogpt` | Muon | `nanogpt_muon` |

Shared demo overrides (fast smoke test):

- `trainer.n_epochs=2`, `trainer.epoch_len=50`
- `datasets.train.download_limit=2000`, `datasets.val.limit=200`
- `writer.mode=offline` (no W&B login required)

In [ ]:
ARCHITECTURES = ["lstm", "transformer", "nanogpt"]
OPTIMIZERS = ["adamw", "muon"]

DEMO_OVERRIDES = [
    "trainer.n_epochs=2",
    "trainer.epoch_len=50",
    "trainer.override=true",
    "datasets.train.download_limit=2000",
    "datasets.val.limit=200",
    "writer.mode=offline",
    "batch_size=16",
    "num_workers=2",
]

EXPERIMENTS = [
    {
        "config_name": arch,
        "optimizer": opt,
        "run_name": f"{arch}_{opt}",
    }
    for arch in ARCHITECTURES
    for opt in OPTIMIZERS
]

print(f"Total runs: {len(EXPERIMENTS)}")
for i, exp in enumerate(EXPERIMENTS, 1):
    print(f"  {i}. {exp['run_name']}")

In [ ]:
def run_experiment(config_name: str, optimizer: str, run_name: str) -> int:
    """Run one training job and return exit code."""
    cmd = [
        "uv", "run", "train.py",
        f"--config-name={config_name}",
        f"optimizer={optimizer}",
        f"writer.run_name={run_name}",
        *DEMO_OVERRIDES,
    ]
    print("\n" + "=" * 72)
    print("Command:", " ".join(cmd))
    print("=" * 72)
    result = subprocess.run(cmd, cwd=CLONE_DIR)
    return result.returncode

In [ ]:
results = {}

for exp in EXPERIMENTS:
    name = exp["run_name"]
    code = run_experiment(
        config_name=exp["config_name"],
        optimizer=exp["optimizer"],
        run_name=name,
    )
    results[name] = "ok" if code == 0 else f"failed (exit {code})"

print("\nSummary")
print("-" * 40)
for name, status in results.items():
    print(f"{name:25s} {status}")

## Inspect saved runs

Checkpoints and logs are written under `saved/<run_name>/`. After training, you can list runs and open the Hydra config:

In [ ]:
saved_root = CLONE_DIR / "saved"
if saved_root.exists():
    for run_dir in sorted(saved_root.iterdir()):
        if run_dir.is_dir():
            config_path = run_dir / "config.yaml"
            checkpoints = list(run_dir.glob("*.pth"))
            print(f"{run_dir.name}: config={'yes' if config_path.exists() else 'no'}, checkpoints={len(checkpoints)}")
else:
    print("No saved runs yet.")

## Suggested follow-up experiments

After the demo grid, try these (from the project README):

1. **Learning-rate sweep** — find best LR per (architecture, optimizer):
   ```bash
   uv run train.py --config-name=lstm optimizer=adamw optimizer.adam.lr=3e-4
   uv run train.py --config-name=lstm optimizer=muon optimizer.muon.lr=0.02
   ```

2. **Muon projection method** — exact SVD vs Newton–Schulz:
   ```bash
   uv run train.py --config-name=transformer optimizer=muon optimizer.muon.projection=exact
   uv run train.py --config-name=transformer optimizer=muon optimizer.muon.projection=ns
   ```

3. **Momentum ablation**:
   ```bash
   uv run train.py --config-name=lstm optimizer=muon optimizer.muon.momentum=0.9 optimizer.muon.nesterov=true
   ```

4. **Matrix metrics** — compare `condition_number_weighted_mean`, `orthogonality_error_weighted_mean`, `spectral_norm_weighted_mean` in logs between AdamW and Muon.

5. **Scale up** — increase data and training length:
   ```bash
   uv run train.py --config-name=nanogpt optimizer=muon \
     datasets.train.download_limit=50000 trainer.n_epochs=20 trainer.epoch_len=500
   ```

6. **Model size** — edit `src/configs/model/*.yaml` (hidden dim, layers, NanoGPT `n_layer` / `n_embd`).

7. **Online logging** — set `writer.mode=online` and configure W&B (`writer.project_name`, `writer.entity`).